# 🚀 Master Training Notebook: Multi-LoRA (SFT + DPO) & NLI Verifier
**Tự động 100%: Clone GitHub + Đồng bộ Google Drive 2 chiều (Không cần tải file thủ công)**

Notebook này thực hiện trọn gói:
1. **Auto-Clone GitHub** (`https://github.com/ducmanh-jr/Multi-Agent_RAG.git`) để lấy mã nguồn và bộ dữ liệu huấn luyện `training_lab/kaggle_fn_v2/datasets/`.
2. **Kết nối Google Drive / GDown**: Tự động tải dữ liệu/checkpoint từ Google Drive (nếu có) và tự động lưu gói `weights_bundle_v2.zip` lên Drive sau khi huấn luyện xong.
3. **Huấn luyện 3 QLoRA Adapters (`router`, `generator`, `critic`)** trên `Qwen/Qwen2.5-7B-Instruct` với **SFT + DPO ($\beta=0.1$)** và **Hard Negatives**.
4. **Huấn luyện NLI Cross-Encoder (`fact_checker_nli`)** và đóng gói `weights_bundle_v2.zip`.


In [ ]:
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN & TỰ ĐỘNG CLONE DỰ ÁN TỪ GITHUB
!pip install -q -U torch transformers peft trl bitsandbytes datasets accelerate sentencepiece gdown huggingface_hub pymupdf

import os
import subprocess

REPO_URL = "https://github.com/ducmanh-jr/Multi-Agent_RAG.git"
PROJECT_DIR = "/content/Multi-Agent_RAG" if os.path.exists("/content") else "/kaggle/working/Multi-Agent_RAG"

if not os.path.exists(PROJECT_DIR):
    print(f"[GitHub] Đang clone dự án từ {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    print(f"[GitHub] Đang cập nhật (git pull) tại {PROJECT_DIR} ...")
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=False)

os.chdir(PROJECT_DIR)
print("[OK] Thư mục làm việc hiện tại:", os.getcwd())


In [ ]:
# BƯỚC 2: KẾT NỐI GOOGLE DRIVE (COLAB MOUNT HOẶC KAGGLE GDOWN)
# - Nếu chạy trên Google Colab: Tự động mount Google Drive để đọc/ghi trực tiếp.
# - Nếu chạy trên Kaggle: Dùng gdown tải từ Drive Folder/File ID (không cần upload thủ công).

DRIVE_WEIGHTS_FILE_ID = ""  # (Tùy chọn) Điền File ID của weights_bundle.zip trên Google Drive nếu muốn fine-tune tiếp
DRIVE_SAVE_DIR = "/content/drive/MyDrive/MultiAgent_RAG_Weights"

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
    IN_COLAB = True
    print(f"[Google Drive] Đã mount thành công! Trọng số sẽ tự động lưu vào: {DRIVE_SAVE_DIR}")
except ImportError:
    print("[Kaggle Mode] Đang chạy trên Kaggle GPU. Sử dụng gdown & /kaggle/working để đồng bộ.")

if DRIVE_WEIGHTS_FILE_ID:
    import gdown
    os.makedirs("training_lab/weights_drive_bundle", exist_ok=True)
    gdown.download(id=DRIVE_WEIGHTS_FILE_ID, output="training_lab/weights_drive_bundle/weights_bundle.zip", quiet=False)
    !unzip -q -o training_lab/weights_drive_bundle/weights_bundle.zip -d training_lab/weights_drive_bundle/
    print("[Google Drive] Đã tải trọng số nền từ Google Drive!")


In [ ]:
# BƯỚC 3: TỰ ĐỘNG SINH BỘ DỮ LIỆU SFT + DPO + HARD NEGATIVES TỪ KHO PDF DỰ ÁN
# Tự động quét toàn bộ PDF trong main/uploads/ để tạo 5 tập dữ liệu huấn luyện chuẩn xác
!python training_lab/kaggle_fn_v2/01_generate_sft_dpo_dataset.py


In [ ]:
# BƯỚC 4: HUẤN LUYỆN 3 LORA ADAPTERS (ROUTER + GENERATOR SFT/DPO + CRITIC HARD-NEGATIVES) & NLI
# Kết quả đầu ra lưu tại: training_lab/kaggle_fn_v2/outputs/ (gồm router/, generator/, critic/, fact_checker_nli/)
!python training_lab/kaggle_fn_v2/02_train_multilora_sft_dpo.py     --dataset-dir training_lab/kaggle_fn_v2/datasets     --output-dir training_lab/kaggle_fn_v2/outputs     --epochs 2     --batch-size 2     --grad-accum 4


In [ ]:
# BƯỚC 5: ĐÓNG GÓI WEIGHTS_BUNDLE.ZIP & TỰ ĐỘNG ĐỒNG BỘ LÊN GOOGLE DRIVE / HUGGINGFACE HUB
import shutil

BUNDLE_PATH = os.path.join(PROJECT_DIR, "weights_bundle_v2")
shutil.make_archive(BUNDLE_PATH, "zip", "training_lab/kaggle_fn_v2/outputs")
zip_file = f"{BUNDLE_PATH}.zip"
print(f"[OK] Đã đóng gói trọng số: {zip_file} ({os.path.getsize(zip_file)/1024/1024:.2f} MB)")

if IN_COLAB and os.path.exists(DRIVE_SAVE_DIR):
    dest = os.path.join(DRIVE_SAVE_DIR, "weights_bundle_v2.zip")
    shutil.copy2(zip_file, dest)
    print(f"[Google Drive] Đã tự động sao lưu lên Google Drive: {dest}")
else:
    print(f"[Kaggle Output] File {zip_file} đã sẵn sàng trong tab Output của Kaggle Notebook!")
